<a href="https://colab.research.google.com/github/furbyhaxx/CLM/blob/main/notebooks/CLM_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating CLM on Colab, without vLLM

Two evaluations, each using only the compute it needs:

1. **DeepSWE best-of-4 verifier** (the README's 31/38 = 81.6% result). The step embeddings are
   published, so this scores them with the projection head alone: no encoder, a few minutes on
   any GPU, or even on CPU.
2. **Typed decisions** ([`LocalLLaMA/typed-decisions`](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)),
   end to end. Every state and option text goes through **Qwen3-8B loaded with Unsloth** in this
   process, then through the heads, exactly as `clm-serve` would answer it. It compares the
   zero-shot reference head, the `clm-raw` ablation and any head you fine-tuned.

A free T4 runs both. On a T4 the encoder is 4-bit; the heads were trained on 16-bit embeddings,
so use an L4 or A100 for exact numbers in part 2. Part 1 is unaffected.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# The CLM repo: the package (heads, Engine, the in-process encoder) and the train / eval scripts.
# It is put on sys.path rather than pip-installed: `contrastive-lm` depends on vLLM, which these
# notebooks do not use.
import os, sys, subprocess
CLM_REPO = "https://github.com/furbyhaxx/CLM"
CLM_REF = "main"
if os.path.isfile(os.path.join("..", "src", "clm", "__init__.py")):   # opened from a clone's notebooks/
    CLM_DIR = os.path.abspath("..")
else:
    CLM_DIR = os.path.abspath("CLM")
    if not os.path.isdir(CLM_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", CLM_REF, CLM_REPO, CLM_DIR], check=True)
for sub in ("src", "train", "preprocessing"):
    sys.path.insert(0, os.path.join(CLM_DIR, sub))
print("CLM:", CLM_DIR)

In [ ]:
# @title GPU
import torch
gpu = torch.cuda.get_device_properties(0)
GPU_GB = gpu.total_memory / 2**30
print(f"GPU = {gpu.name}, {GPU_GB:.1f} GB, bf16 = {torch.cuda.is_bf16_supported()}")

## 1. DeepSWE: CLM as a best-of-N verifier

Each step of each rollout is a (state, action) pair. The head scores every step, the trajectory
score is the mean of its final 12 step scores, and the verifier picks the best of N=4 rollouts
per task. `evaluation/bon_eval.py` computes the exact expectation over ties and subsets.

The head [`Contrastive-LM/deepswe-clm-heads-8k`](https://huggingface.co/Contrastive-LM/deepswe-clm-heads-8k)
was trained on 75 tasks and is evaluated on the 38 held-out ones. The evaluation embeddings
(Claude Opus 5 rollouts, 4 per task, 44k steps, ~0.7 GB) are on the Hub, so the 8K-token
DeepSWE states never need re-encoding.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

HEADS_DIR = snapshot_download("Contrastive-LM/deepswe-clm-heads-8k", local_dir="heads/deepswe")

# the README's dataset id, else its public mirror (same parquet: 44,409 steps, 449 rollouts, 113 tasks)
for EVAL_DATASET in ("Contrastive-LM/deepswe-clm-embeddings-8k", "tarsur385/deepswe-prm-embeddings-8k"):
    try:
        HfApi().dataset_info(EVAL_DATASET); break
    except Exception:
        continue
print("eval embeddings:", EVAL_DATASET)

In [ ]:
!python {CLM_DIR}/evaluation/bon_eval.py --hf-dataset {EVAL_DATASET} \
    --checkpoint {HEADS_DIR}/best_head.pt --tasks-file {HEADS_DIR}/heldout_tasks.json \
    --n 4 --window 12 --output results/deepswe_bo4.json

In [ ]:
import json
res = json.load(open("results/deepswe_bo4.json"))
sel = next(iter(res["selectors"].values()))
print(f"CLM best-of-4 {sel['rate']:.1%} ({sel['resolved']}/{sel['n_tasks']}) | "
      f"random pick {res['random_pick']:.1%} | oracle {res['oracle_any']:.1%}")
# published: 31/38 = 81.6%, pass@1 73.7%, oracle 89.5%

Heads you train with `CLM_Finetune.ipynb` (part 2) are evaluated the same way: point `--checkpoint`
at `runs/deepswe/best_head.pt`, or use `--fold-spec runs/deepswe/fold_spec.json` for fold-wise heads.

## 2. Typed decisions, end to end with the Unsloth encoder

Every test row has a state, several typed questions and a gold annotator distribution per
question. We report:

* **accuracy**: argmax answer == gold label,
* **soft CE**: cross-entropy of the predicted distribution against the annotators' distribution
  (lower is better),
* a **majority** baseline, which answers each question id with its most frequent training label.

In [ ]:
import unsloth  # before transformers
from clm import Engine
from clm.heads import download
from clm.unsloth_embedder import UnslothEmbedder

ENCODER = "Qwen/Qwen3-8B"  # or the LoRA adapter dir saved by CLM_Finetune.ipynb (part 3)
LOAD_IN_4BIT = None        # None = auto: 4-bit below 20 GB of GPU memory
DATA, WORKFLOW = "LocalLLaMA/typed-decisions", "all"   # or one workflow, e.g. "customer_service"

# Heads to compare: the reference head plus the fine-tuned ones that match ENCODER. A head only makes
# sense with the encoder it was trained against: a LoRA encoder brings its own head, and DeepSWE heads
# are trajectory verifiers, not typed-question heads.
import glob
HEADS = {"clm-latest": download()}
if os.path.isdir(ENCODER):
    HEADS["ft/lora"] = os.path.join(os.path.dirname(os.path.abspath(ENCODER)), "best_head.pt")
else:
    HEADS.update({f"ft/{os.path.basename(os.path.dirname(p))}": p for p in sorted(glob.glob("runs/*/best_head.pt"))
                  if not any(k in p for k in ("deepswe", "lora"))})

embedder = UnslothEmbedder(ENCODER, max_tokens=2048, load_in_4bit=LOAD_IN_4BIT)
engine = Engine(embedder=embedder, checkpoint=HEADS.pop("clm-latest"), models=HEADS)
MODELS = [m["name"] for m in engine.models()]
print(MODELS)

In [ ]:
import math, time
from collections import Counter, defaultdict
import adapters
from finetune import load_typed_rows
from clm.schema import build_pairs, label_of, probabilities_of

train_rows = load_typed_rows(DATA, "train", WORKFLOW, None)
test_rows = load_typed_rows(DATA, "test", WORKFLOW, None)

def parse(row):
    return tuple(adapters._maybe_json(row[k]) for k in ("state", "questions", "gold"))

# Embed every test text once, in one length-sorted, token-batched pass; the answers below hit the cache.
texts = [t for r in test_rows for (s, _, c) in build_pairs(*parse(r)[:2]).values() for t in (s, *c)]
t0 = time.time()
_, tokens = embedder.embed(texts)
print(f"{len(set(texts))} unique texts, {tokens:,} tokens in {time.time() - t0:.0f}s")

In [ ]:
def gold_of(g):
    label = adapters._key_of(g["label"])
    probs = {adapters._key_of(k): float(v) for k, v in (g.get("probabilities") or {}).items()} or {label: 1.0}
    z = sum(probs.values())
    return label, {k: v / z for k, v in probs.items()}

def evaluate(model):
    tot, per_wf, per_q = [0, 0, 0.0], defaultdict(lambda: [0, 0]), defaultdict(lambda: [0, 0])
    for r in test_rows:
        state, questions, gold = parse(r)
        answers = engine.answer(state, questions, model=model)["answers"]
        for qid, g in gold.items():
            if qid not in answers or "label" not in g:
                continue
            label, target = gold_of(g)
            a = answers[qid]
            ok = label_of(a) == label
            p = probabilities_of(a)
            tot[0] += ok; tot[1] += 1
            tot[2] -= sum(v * math.log(max(p.get(k, 0.0), 1e-12)) for k, v in target.items())
            for d, k in ((per_wf, r.get("workflow")), (per_q, qid)):
                d[k][0] += ok; d[k][1] += 1
    return {"acc": tot[0] / tot[1], "soft_ce": tot[2] / tot[1], "n": tot[1],
            "per_workflow": {k: round(h / n, 3) for k, (h, n) in sorted(per_wf.items())},
            "per_question": {k: round(h / n, 3) for k, (h, n) in sorted(per_q.items())}}

def majority():
    counts = defaultdict(Counter)
    for r in train_rows:
        for qid, g in parse(r)[2].items():
            if "label" in g:
                counts[qid][adapters._key_of(g["label"])] += 1
    hit = n = 0
    for r in test_rows:
        for qid, g in parse(r)[2].items():
            if "label" in g and counts[qid]:
                hit += counts[qid].most_common(1)[0][0] == adapters._key_of(g["label"]); n += 1
    return hit / n

results = {m: evaluate(m) for m in MODELS}
print(f"{'model':<24}{'acc':>8}{'soft CE':>10}")
print(f"{'majority (train)':<24}{majority():>8.3f}")
for m, r in results.items():
    print(f"{m:<24}{r['acc']:>8.3f}{r['soft_ce']:>10.3f}")

In [ ]:
# per-workflow / per-question accuracy
import pandas as pd
pd.DataFrame({m: {**r["per_workflow"], **{f"q:{k}": v for k, v in r["per_question"].items()}}
              for m, r in results.items()})

In [ ]:
os.makedirs("results", exist_ok=True)
json.dump({"encoder": ENCODER, "encoder_4bit": embedder.encoder.load_in_4bit,
           "data": DATA, "workflow": WORKFLOW, "results": results},
          open(f"results/typed_decisions_{WORKFLOW}.json", "w"), indent=1)
print("wrote", f"results/typed_decisions_{WORKFLOW}.json")

`clm-latest` is zero-shot here: the reference head never saw these workflows. The fine-tuned rows
show how much a few minutes of head training adds; see **CLM_Finetune.ipynb**.